# Collaborative Filtering example 

In [20]:
import pandas as pd 
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse 

class CF(object):
    def __init__(self, Y_data, k, dist_func=cosine_similarity, uuCF=1):
        self.uuCF = uuCF  # user-user (1) or item-item (0)
        self.Y_data = Y_data if uuCF else Y_data[:, [1, 0, 2]]
        self.k = k
        self.dist_func = dist_func
        self.Ybar_data = None

        self.n_users = int(np.max(self.Y_data[:, 0])) + 1 
        self.n_items = int(np.max(self.Y_data[:, 1])) + 1
    
    def add(self, new_data):
        self.Y_data = np.concatenate((self.Y_data, new_data), axis=0)
    
    def normalize_Y(self):
        users = self.Y_data[:, 0]
        self.Ybar_data = self.Y_data.copy()
        self.mu = np.zeros((self.n_users,))

        for n in range(self.n_users):  # FIX: xrange → range
            ids = np.where(users == n)[0].astype(np.int32)
            item_ids = self.Y_data[ids, 1]
            ratings = self.Y_data[ids, 2]

            m = np.mean(ratings) 
            if np.isnan(m):
                m = 0

            self.mu[n] = m
            self.Ybar_data[ids, 2] = ratings - self.mu[n]

        # sparse matrix
        self.Ybar = sparse.coo_matrix(
            (self.Ybar_data[:, 2],
             (self.Ybar_data[:, 1], self.Ybar_data[:, 0])),
            (self.n_items, self.n_users)
        ).tocsr()

    def similarity(self):
        self.S = self.dist_func(self.Ybar.T, self.Ybar.T)
        
    def refresh(self):
        self.normalize_Y()
        self.similarity()
        
    def fit(self):
        self.refresh()
        
    def __pred(self, u, i, normalized=1):
        ids = np.where(self.Y_data[:, 1] == i)[0].astype(np.int32)
        users_rated_i = self.Y_data[ids, 0].astype(np.int32)

        sim = self.S[u, users_rated_i]

        # top k similar users
        a = np.argsort(sim)[-self.k:]
        nearest_s = sim[a]

        r = self.Ybar[i, users_rated_i[a]].toarray().flatten()

        if normalized:
            return (r @ nearest_s) / (np.abs(nearest_s).sum() + 1e-8)

        return (r @ nearest_s) / (np.abs(nearest_s).sum() + 1e-8) + self.mu[u]
    
    def pred(self, u, i, normalized=1):
        if self.uuCF:
            return self.__pred(u, i, normalized)
        return self.__pred(i, u, normalized)
            
    def recommend(self, u):
        ids = np.where(self.Y_data[:, 0] == u)[0]
        items_rated_by_u = self.Y_data[ids, 1].tolist()              
        recommended_items = []

        for i in range(self.n_items):  # FIX
            if i not in items_rated_by_u:
                rating = self.__pred(u, i)
                if rating > 0: 
                    recommended_items.append(i)
        
        return recommended_items 

    def print_recommendation(self):
        print('Recommendation:')
        for u in range(self.n_users):  # FIX
            recommended_items = self.recommend(u)
            if self.uuCF:
                print('Recommend item(s):', recommended_items, 'for user', u)
            else: 
                print('Recommend item', u, 'for user(s):', recommended_items)

In [43]:
# data file 
r_cols = ['user_id', 'item_id', 'rating']
ratings = pd.read_csv('ex.dat', sep = ' ', names = r_cols, encoding='latin-1')
Y_data = ratings.to_numpy()

rs = CF(Y_data, k = 9, uuCF = 1)
rs.fit()

rs.print_recommendation()

Recommendation:
Recommend item(s): [2] for user 0
Recommend item(s): [1] for user 1
Recommend item(s): [] for user 2
Recommend item(s): [4] for user 3
Recommend item(s): [4] for user 4
Recommend item(s): [4] for user 5
Recommend item(s): [] for user 6


In [45]:
rs = CF(Y_data, k = 2, uuCF = 0)
rs.fit()

rs.print_recommendation()

Recommendation:
Recommend item 0 for user(s): []
Recommend item 1 for user(s): [1]
Recommend item 2 for user(s): [0]
Recommend item 3 for user(s): [5]
Recommend item 4 for user(s): [3, 4, 5]


## MovieLens

In [24]:
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']

ratings_base = pd.read_csv('ml-100k/ub.base', sep='\t', names=r_cols, encoding='latin-1')
ratings_test = pd.read_csv('ml-100k/ub.test', sep='\t', names=r_cols, encoding='latin-1')

rate_train = ratings_base.to_numpy()
rate_test = ratings_test.to_numpy()

# indices start from 0
rate_train[:, :2] -= 1
rate_test[:, :2] -= 1

In [ ]:
rs = CF(rate_train, k = 30, uuCF = 1)
rs.fit()

n_tests = rate_test.shape[0]
SE = 0 # squared error
for n in range(n_tests):
    pred = rs.pred(rate_test[n, 0], rate_test[n, 1], normalized = 0)
    SE += (pred - rate_test[n, 2])**2 

RMSE = np.sqrt(SE/n_tests)
print ('User-user CF, Sai số bình phương =', RMSE)

User-user CF, RMSE = 0.9951644420644284


In [27]:
rs = CF(rate_train, k = 30, uuCF = 0)
rs.fit()

n_tests = rate_test.shape[0]
SE = 0 # squared error
for n in range(n_tests):
    pred = rs.pred(rate_test[n, 0], rate_test[n, 1], normalized = 0)
    SE += (pred - rate_test[n, 2])**2 

RMSE = np.sqrt(SE/n_tests)
print ('Item-item CF, Sai số bình phương =', RMSE)

Item-item CF, Sai số bình phương = 0.9867689046132182


In [48]:
rs = CF(rate_train, k =100000, uuCF = 0)
rs.fit()

n_tests = rate_test.shape[0]
SE = 0 # squared error
for n in range(n_tests):
    pred = rs.pred(rate_test[n, 0], rate_test[n, 1], normalized = 0)
    SE += (pred - rate_test[n, 2])**2 

RMSE = np.sqrt(SE/n_tests)
print ('Item-item CF, Sai số bình phương =', RMSE)

Item-item CF, Sai số bình phương = 0.9991333317848617


In [ ]:
# data file 
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv('ml-100k/ub.base', sep='\t', names=r_cols, encoding='latin-1')
Y_data = ratings.to_numpy()

rs = CF(Y_data, k = 943, uuCF = 1)
rs.fit()

rs.print_recommendation()

Recommendation:
Recommend item(s): [] for user 0
Recommend item(s): [] for user 1
Recommend item(s): [] for user 2
Recommend item(s): [] for user 3
Recommend item(s): [] for user 4
Recommend item(s): [] for user 5
Recommend item(s): [] for user 6
Recommend item(s): [] for user 7
Recommend item(s): [] for user 8
Recommend item(s): [] for user 9
Recommend item(s): [] for user 10
Recommend item(s): [] for user 11
Recommend item(s): [] for user 12
Recommend item(s): [] for user 13
Recommend item(s): [] for user 14
Recommend item(s): [] for user 15
Recommend item(s): [] for user 16
Recommend item(s): [] for user 17
Recommend item(s): [] for user 18
Recommend item(s): [] for user 19
Recommend item(s): [] for user 20
Recommend item(s): [] for user 21
Recommend item(s): [] for user 22
Recommend item(s): [1] for user 23
Recommend item(s): [] for user 24
Recommend item(s): [] for user 25
Recommend item(s): [] for user 26
Recommend item(s): [] for user 27
Recommend item(s): [] for user 28
Recomme